# 📓 Notebook 3 — Gradio 6.0 Interactive Frontend
### Egyptian Telecom RAG Support Assistant

This notebook:
1. Connects to the FastAPI backend from Notebook 2 (via its localtunnel URL)
2. Builds a Gradio Blocks UI with a messages-based Chatbot, operator dropdown, quick-action buttons, and a satisfaction survey
3. Launches a public Gradio share link

**Before running:** make sure Notebook 2 is running and copy its localtunnel URL into `BACKEND_URL` below.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q gradio==6.0.0 requests==2.32.3

## Cell 2 — Configuration

In [ ]:
# 👇 PASTE the localtunnel URL printed by Notebook 2, Cell 6, here:
BACKEND_URL = "https://your-tunnel-url.loca.lt"  # e.g. https://abc-def-123.loca.lt

QUERY_ENDPOINT = f"{BACKEND_URL}/query"
HEALTH_ENDPOINT = f"{BACKEND_URL}/health"

# localtunnel shows an HTML "friendly reminder" interstitial page to any client that
# hasn't confirmed it yet, instead of proxying straight to the API. requests.json()
# then fails trying to parse that HTML as JSON (e.g. "Extra data: line 1 column 5").
# Sending this header on every request tells localtunnel to skip the interstitial.
TUNNEL_HEADERS = {"Bypass-Tunnel-Reminder": "true"}

import requests

try:
    resp = requests.get(HEALTH_ENDPOINT, headers=TUNNEL_HEADERS, timeout=10)
    resp.raise_for_status()
    print("✅ Backend reachable:", resp.json())
except requests.exceptions.JSONDecodeError:
    print("⚠️ Backend responded, but not with JSON. Raw response below — this usually")
    print("   means localtunnel's interstitial page slipped through, or BACKEND_URL")
    print("   is wrong. First 300 chars of response:")
    print(resp.text[:300])
except Exception as e:
    print("⚠️ Could not reach backend. Double-check BACKEND_URL. Error:", e)

## Cell 3 — Feedback logging setup

In [ ]:
import csv
import os
from datetime import datetime

FEEDBACK_LOG_PATH = "/content/feedback_log.csv"

if not os.path.exists(FEEDBACK_LOG_PATH):
    with open(FEEDBACK_LOG_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "rating", "feedback_text", "company"])

def log_feedback(rating: int, feedback_text: str, company: str):
    with open(FEEDBACK_LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([datetime.utcnow().isoformat(), rating, feedback_text, company])
    return f"✅ Thank you! Your {rating}-star feedback has been recorded."

print("✅ Feedback logging ready at:", FEEDBACK_LOG_PATH)

## Cell 4 — Backend call helper

In [ ]:
import requests

def call_rag_backend(question: str, company: str) -> tuple[str, str]:
    """Calls the FastAPI backend and returns (answer, formatted_sources)."""
    try:
        response = requests.post(
            QUERY_ENDPOINT,
            json={"question": question, "company": company},
            headers=TUNNEL_HEADERS,  # bypasses localtunnel's HTML interstitial page
            timeout=60
        )
        response.raise_for_status()
        data = response.json()

        answer = data.get("answer", "Sorry, I couldn't generate an answer.")
        sources = data.get("sources", [])

        if sources:
            sources_text = "\n".join(
                f"- [{s['company']}] {s['source']}" for s in sources
            )
        else:
            sources_text = "No specific source documents matched."

        return answer, sources_text

    except requests.exceptions.Timeout:
        return "⏱️ The request timed out. Please try again.", ""
    except requests.exceptions.JSONDecodeError:
        return "⚠️ Backend returned a non-JSON response. Check that BACKEND_URL is correct and Notebook 2 is still running.", ""
    except requests.exceptions.RequestException as e:
        return f"⚠️ Could not reach the support backend: {e}", ""

## Cell 5 — Gradio Blocks UI

In [ ]:
import gradio as gr

custom_css = """
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    max-width: 1100px !important;
    margin: auto !important;
}
#header-banner {
    background: linear-gradient(90deg, #6a11cb 0%, #2575fc 100%);
    padding: 20px;
    border-radius: 12px;
    color: white;
    text-align: center;
    margin-bottom: 15px;
}
#quick-actions button {
    border-radius: 20px !important;
}
.sources-box textarea {
    font-size: 12px !important;
    color: #444 !important;
}
"""

QUICK_ACTIONS = {
    "🔄 Reset Router": "How do I reset my home router to factory settings?",
    "💰 Mobile Wallet Activation": "How do I activate my mobile wallet?",
    "📶 Check Data Quota": "How can I check my remaining internet data quota?",
    "💳 Recharge Balance": "What are the ways to recharge my balance?",
    "📄 Bill Inquiry": "How do I view or pay my monthly bill?",
    "🌐 No Internet Connection": "My internet connection is not working, what should I do?",
}

def user_submit(message, history, company):
    if not message or not message.strip():
        return history, "", ""
    history = history + [{"role": "user", "content": message}]
    answer, sources_text = call_rag_backend(message, company)
    history = history + [{"role": "assistant", "content": answer}]
    return history, "", sources_text

def quick_action_submit(action_label, history, company):
    question = QUICK_ACTIONS[action_label]
    return user_submit(question, history, company)

def clear_chat():
    return [], "", ""

def submit_feedback(rating, feedback_text, company):
    if rating is None:
        return "⚠️ Please select a star rating before submitting."
    return log_feedback(int(rating), feedback_text or "", company)

with gr.Blocks(css=custom_css, title="Egyptian Telecom Support Assistant") as demo:

    gr.HTML("""
    <div id="header-banner">
        <h1>📞 Egyptian Telecom Support Assistant</h1>
        <p>Ask about WE, Vodafone, or Etisalat services — router setup, wallet, billing, and more.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=220):
            company_selector = gr.Dropdown(
                choices=["All", "WE", "Vodafone", "Etisalat"],
                value="All",
                label="🏢 Select Operator",
                interactive=True
            )

            gr.Markdown("### Quick Actions")
            with gr.Column(elem_id="quick-actions"):
                quick_buttons = []
                for label in QUICK_ACTIONS:
                    btn = gr.Button(label, size="sm")
                    quick_buttons.append((btn, label))

            gr.Markdown("### 📚 Sources Used")
            sources_display = gr.Textbox(
                label="",
                lines=6,
                interactive=False,
                elem_classes="sources-box"
            )

            clear_btn = gr.Button("🗑️ Clear Conversation")

        with gr.Column(scale=3):
            # NOTE: Gradio 6.0 removed the old "tuples" chat format entirely and made
            # "messages" (role/content dicts) the only supported format — as part of
            # that change, the `type` parameter itself was removed from gr.Chatbot's
            # constructor (there's no longer a format to choose between). Passing
            # type="messages" now raises: TypeError: unexpected keyword argument 'type'.
            # Our chat history already uses role/content dicts throughout this file,
            # which is exactly what Gradio 6 expects by default — no argument needed.
            chatbot = gr.Chatbot(
                label="Support Chat",
                height=480,
                avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/4712/4712109.png")
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Type your question here (Arabic or English)...",
                    scale=4,
                    show_label=False
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

    gr.Markdown("---")
    gr.Markdown("### ⭐ Rate Your Experience")
    with gr.Row():
        with gr.Column(scale=2):
            rating_radio = gr.Radio(
                choices=["1", "2", "3", "4", "5"],
                label="How satisfied were you with this support session?",
                value=None
            )
            feedback_text = gr.Textbox(
                placeholder="Optional: tell us more about your experience...",
                label="Additional Feedback",
                lines=2
            )
            feedback_submit_btn = gr.Button("Submit Feedback", variant="secondary")
        with gr.Column(scale=1):
            feedback_status = gr.Textbox(label="Status", interactive=False)

    # Wire up events
    send_btn.click(
        fn=user_submit,
        inputs=[msg_box, chatbot, company_selector],
        outputs=[chatbot, msg_box, sources_display]
    )
    msg_box.submit(
        fn=user_submit,
        inputs=[msg_box, chatbot, company_selector],
        outputs=[chatbot, msg_box, sources_display]
    )

    for btn, label in quick_buttons:
        btn.click(
            fn=lambda history, company, lbl=label: quick_action_submit(lbl, history, company),
            inputs=[chatbot, company_selector],
            outputs=[chatbot, msg_box, sources_display]
        )

    clear_btn.click(fn=clear_chat, outputs=[chatbot, msg_box, sources_display])

    feedback_submit_btn.click(
        fn=submit_feedback,
        inputs=[rating_radio, feedback_text, company_selector],
        outputs=[feedback_status]
    )

print("✅ Gradio UI defined")

## Cell 6 — Launch

In [ ]:
demo.launch(share=True, css=custom_css)